# PyG HeteroData export

`to_pyg` exports an AnnNet graph to PyTorch Geometric
`HeteroData` for graph neural network workflows.


In [ ]:
import annnet as an

an.info()


## Build a heterogeneous graph


In [ ]:
G = an.AnnNet(directed=True)
G.add_vertices('p1', kind='protein', activity=1.2, abundance=4.0)
G.add_vertices('p2', kind='protein', activity=-0.4, abundance=2.5)
G.add_vertices('g1', kind='gene', expression=8.0, length=1200.0)
G.add_vertices('drug_a', kind='drug', dosage=10.0, approved=1.0)

G.add_edges('p1', 'g1', edge_id='regulates', weight=0.8)
G.add_edges('p2', 'g1', edge_id='represses', weight=-0.6)
G.add_edges('drug_a', 'p1', edge_id='targets', weight=0.5)
G.add_edges('drug_a', 'p2', edge_id='off_target', weight=0.25)
G.slices.add('train')
G.slices.add_edges('train', ['regulates', 'represses'])

G.views.vertices()


## Draw the source graph


In [ ]:
from annnet.utils import plotting

plotting.plot(G, backend='graphviz', show_edge_labels=True)


## Export to PyG and run a tensor operation


In [ ]:
import torch

data = an.to_pyg(
    G,
    node_features={
        'protein': ['activity', 'abundance'],
        'gene': ['expression', 'length'],
        'drug': ['dosage', 'approved'],
    },
    slice_id='train',
    hyperedge_mode='skip',
)
data.validate(raise_on_error=True)

protein_activity = data['protein'].x[:, 0]
activity_probability = torch.sigmoid(protein_activity)
homogeneous = data.to_homogeneous()

print(data)
print('protein activity:', protein_activity.tolist())
print('sigmoid(activity):', activity_probability.tolist())
print('homogeneous edge_index shape:', tuple(homogeneous.edge_index.shape))


AnnNet remains useful before tensorization: it stores names,
metadata, slices, and graph semantics. `to_pyg` is the boundary
where selected numeric attributes become tensors.
